# Experience 2: quantization of the FINE-TUNED Qwen2.5-Coder-1.5B (fp16 / INT8 / nf4)
Same code as `quantization-qwen.ipynb` (same generate(), same Part A and Part B), plus Part C (the 9 security prompts).
Pipeline: base + LoRA adapter -> merged in fp16 -> quantized at load time (INT8 / nf4 with the SAME BitsAndBytesConfig as the base experiment).

Inputs to add (one Kaggle Dataset is enough): `adapter_run1.zip` and the 4 reference files `ft_ref_benchmark.csv`, `ft_ref_humaneval.json`, `ft_ref_vuln.csv`, `base_ref_vuln.csv`.

Run 1: `MODEL_KIND = "ft"`, `PARTS = ("A","B","C")`. Optional run 2: `MODEL_KIND = "base"`, `PARTS = ("C",)` (security prompts for the base model under quantization).

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO_URL = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"
BRANCH = "main"

if not os.path.exists("../config.json"):
    !git clone --depth 1 -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

Cloning into 'SLM-Software-Engineering'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 104 (delta 31), reused 64 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 282.64 KiB | 16.63 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/kaggle/working/SLM-Software-Engineering/benchmarking/notebooks


In [2]:
# torchao 0.10.0 (preinstalled) is rejected by peft 0.21.2 (needs >=0.16); torchao is not used here, so remove it.
!pip uninstall -y -q torchao
!pip install -q -U bitsandbytes
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-7d9b6b8f-18bf-ea13-f91c-9be9b0e770b7)
GPU 1: Tesla T4 (UUID: GPU-8ca6b366-6387-2127-f4d2-234cb2b6a7b7)


In [3]:
# === CHANGE ONLY THESE TWO LINES ===
MODEL_KIND = "ft"                  # "ft" = merged fine-tuned model | "base" = original model
PARTS = ("A", "B", "C")            # A = 15 prompts, B = HumanEval 20, C = 9 security prompts  (base run: ("C",))
# ===================================
assert MODEL_KIND in ("ft", "base")
import json, time, sys, re, gc, platform, shutil, glob, zipfile, hashlib
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

MODEL_KEY = "qwen"
VARIANTS = ["fp16", "int8", "nf4"]           # bf16 of the fine-tuned model was already evaluated (full benchmark)
OVERWRITE = False

with open("../config.json") as f:
    config = json.load(f)
MODEL_NAME = config["models"][MODEL_KEY]
GEN_PARAMS = config["generation_params"]
HE_CONFIG = config["humaneval"]
SYSTEM_PROMPT = config["no_think_models"].get(MODEL_KEY)
model_slug = MODEL_NAME.split("/")[-1].lower()
GPU_NAME = torch.cuda.get_device_name(0)
OUT = f"../results/quant_{MODEL_KIND}"
os.makedirs(OUT, exist_ok=True)
print(MODEL_NAME, "|", GPU_NAME, "|", MODEL_KIND, PARTS)

def norm(s):
    return re.sub(r"\s+", "", str(s))

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

# Reference = the same model in bf16 (already graded). Same text as the reference -> same grade is copied.
if MODEL_KIND == "ft":
    pa, pb, pc = find("ft_ref_benchmark.csv"), find("ft_ref_humaneval.json"), find("ft_ref_vuln.csv")
else:
    pa, pb, pc = None, None, find("base_ref_vuln.csv")
base_a = pd.read_csv(pa).set_index("id") if pa else None
base_b = {d["task_id"]: d["status"] for d in json.load(open(pb))["details"]} if pb else None
base_c = pd.read_csv(pc).set_index("id") if pc else None
print("refs found:", {"A": bool(pa), "B": bool(pb), "C": bool(pc)})
for part, ref in (("A", pa), ("B", pb), ("C", pc)):
    if part in PARTS and MODEL_KIND == "ft" or (part == "C" and "C" in PARTS):
        assert ref, f"reference file for Part {part} not found in /kaggle/input (add the ref files to the dataset)"


Qwen/Qwen2.5-Coder-1.5B-Instruct | Tesla T4 | ft ('A', 'B', 'C')
refs found: {'A': True, 'B': True, 'C': True}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)     # base tokenizer = same prompt format as every other run
MODEL_PATH = MODEL_NAME
ADAPTER_META = None

if MODEL_KIND == "ft":
    from peft import PeftModel
    cfgs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if not cfgs:
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        assert zips, "Add adapter_run1.zip as a Kaggle Input, then rerun"
        zipfile.ZipFile(zips[0]).extractall("/kaggle/working/adapter_run1")
        cfgs = glob.glob("/kaggle/working/adapter_run1/**/adapter_config.json", recursive=True)
    ADAPTER_DIR = os.path.dirname(cfgs[0])
    wfile = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
    acfg = json.load(open(cfgs[0]))
    ADAPTER_META = {"r": acfg.get("r"), "lora_alpha": acfg.get("lora_alpha"),
                    "sha256": hashlib.sha256(open(wfile, "rb").read()).hexdigest()}
    print("adapter:", ADAPTER_META)
    EXPECTED_SHA = "02ec15117c3bb9e0e84b5a0a7eec98f0bf1d14f438d6fa592d56a3cbe90ab9f3"   # adapter used in the bf16 evaluation
    assert ADAPTER_META["sha256"] == EXPECTED_SHA, "adapter differs from the one evaluated in bf16"
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map={"": 0})
    m = PeftModel.from_pretrained(m, ADAPTER_DIR).merge_and_unload()
    MODEL_PATH = "/kaggle/working/merged_ft_fp16"
    m.save_pretrained(MODEL_PATH)
    del m; gc.collect(); torch.cuda.empty_cache()
    print("merged (fp16) model saved to", MODEL_PATH)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

adapter: {'r': 16, 'lora_alpha': 16, 'sha256': '02ec15117c3bb9e0e84b5a0a7eec98f0bf1d14f438d6fa592d56a3cbe90ab9f3'}


... (2 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

merged (fp16) model saved to /kaggle/working/merged_ft_fp16


In [5]:
def load_model(variant):
    kw = {"device_map": {"": 0}}
    if variant == "bf16":
        kw["dtype"] = torch.bfloat16
    elif variant == "fp16":
        kw["dtype"] = torch.float16
    elif variant == "int8":
        kw["dtype"] = torch.float16
        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif variant == "nf4":
        kw["dtype"] = torch.float16
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    else:
        raise ValueError(variant)
    m = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **kw)
    m.eval()
    return m

def generate(model, user_prompt):
    # IDENTIQUE a la fonction du benchmark de base (memes mesures), sauf que le modele est passe en argument
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]

    torch.cuda.synchronize(); t0 = time.time()
    model.generate(**inputs, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize(); ttft = time.time() - t0

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                         do_sample=GEN_PARAMS["do_sample"])
    torch.cuda.synchronize(); latency = time.time() - t0

    new_tokens = out[0][n_in:]
    return {
        "response": tokenizer.decode(new_tokens, skip_special_tokens=True),
        "gpu": GPU_NAME,
        "ttft_s": round(ttft, 3),
        "latency_s": round(latency, 3),
        "tokens_per_sec": round(len(new_tokens) / latency, 2) if latency > 0 else None,
        "input_tokens": n_in,
        "output_tokens": len(new_tokens),
        "gpu_memory_go": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }

In [6]:
with open("../prompts.json") as f:
    prompts = json.load(f)["prompts"]
with open("../humaneval_subset.json") as f:
    subset = json.load(f)
humaneval = load_dataset(HE_CONFIG["dataset"], split="test")
humaneval = humaneval.filter(lambda ex: ex["task_id"] in subset["task_ids"])
HEADER = "from typing import *\nimport math\n\n"
LABEL_COLS = ("bug_label", "note_qualite", "hallucination")

def run_part_a(model, variant):
    rows = []
    for p in prompts:
        gen = generate(model, p["prompt"])
        response = gen.pop("response")
        status, error = None, None
        if p["task"] == "generation_code" and "test" in p:
            status, error = run_tests(extract_code(response), p["test"], timeout=HE_CONFIG["timeout_seconds"])
        elif p["task"] == "completion_code" and "expected" in p:
            status = "pass" if norm(p["expected"]) in norm(response) else "fail"
        same = bool(base_a is not None and p["id"] in base_a.index
                    and norm(base_a.loc[p["id"], "response"]) == norm(response))
        row = {"variant": variant, "id": p["id"], "task": p["task"], "prompt": p["prompt"],
               "response": response, **gen, "auto_status": status, "auto_error": error,
               "same_as_bf16": same}
        # meme texte que la reference -> meme note (on recopie) ; sinon a noter a la main
        for c in LABEL_COLS:
            row[c] = base_a.loc[p["id"], c] if same else None
        rows.append(row)
        print(variant, p["id"], status, "| identique bf16:", same)
    return pd.DataFrame(rows)

def run_part_b(model, variant):
    details = []
    for ex in humaneval:
        gen = generate(model, HE_CONFIG["instruction"] + ex["prompt"])
        code_generated = extract_code(gen["response"])
        test_code = ex["test"] + f"\ncheck({ex['entry_point']})"
        status, error = run_tests(HEADER + ex["prompt"] + "\n" + code_generated, test_code,
                                  timeout=HE_CONFIG["timeout_seconds"])
        details.append({"task_id": ex["task_id"], "status": status, "error": error,
                        "ttft_s": gen["ttft_s"], "latency_s": gen["latency_s"],
                        "tokens_per_sec": gen["tokens_per_sec"], "output_tokens": gen["output_tokens"],
                        "gpu_memory_go": gen["gpu_memory_go"], "response": gen["response"]})
        print(variant, ex["task_id"], status)
    n_pass = sum(d["status"] == "pass" for d in details)
    return {"model": MODEL_NAME, "variant": variant, "gpu": GPU_NAME, "n_problems": len(details),
            "n_pass": n_pass, "pass@1": round(100 * n_pass / len(details), 1), "details": details}

sec = json.load(open("../prompts_securite.json"))["prompts"]

def run_part_c(model, variant):
    rows = []
    for p in sec:
        gen = generate(model, p["prompt"])
        resp = gen.pop("response")
        kw = [k.lower() for k in p.get("keywords", [])]
        same = bool(base_c is not None and p["id"] in base_c.index
                    and norm(base_c.loc[p["id"], "response"]) == norm(resp))
        rows.append({"variant": variant, "id": p["id"], "prompt": p["prompt"], "response": resp, **gen,
                     "has_vuln": p["has_vuln"], "expected_vuln": p["expected_vuln"],
                     "mot_cle_trouve": any(k in resp.lower() for k in kw) if kw else None,
                     "same_as_ref": same,
                     "vuln_label": base_c.loc[p["id"], "vuln_label"] if same else None})
        print(variant, p["id"], "| keyword:", rows[-1]["mot_cle_trouve"], "| same as ref:", same)
    return pd.DataFrame(rows)


README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/164 [00:00<?, ? examples/s]

In [7]:
def summarize(variant, df_a, he, df_c, weights_gb):
    d = {"variant": variant, "model_kind": MODEL_KIND, "weights_gb": round(weights_gb, 2)}
    allg = pd.concat([x for x in (df_a, df_c) if x is not None])
    d.update({"peak_gpu_gb": allg["gpu_memory_go"].max(), "ttft_s": round(allg["ttft_s"].mean(), 3),
              "latency_s": round(allg["latency_s"].mean(), 2), "tokens_per_sec": round(allg["tokens_per_sec"].mean(), 2),
              "output_tokens": round(allg["output_tokens"].mean(), 1)})
    if df_a is not None:
        auto = df_a[df_a["auto_status"].notna()]
        d["auto_pass_P1_P6"] = f"{(auto['auto_status'] == 'pass').sum()}/{len(auto)}"
        d["same_text_as_ref_A_%"] = round(df_a["same_as_bf16"].mean() * 100, 1)
        d["rows_to_grade_A"] = int((~df_a["same_as_bf16"]).sum())
    if he is not None:
        d["humaneval_pass@1_%"] = he["pass@1"]
        d["he_same_status_as_ref_%"] = (sum(base_b.get(x["task_id"]) == x["status"] for x in he["details"]) / len(he["details"]) * 100
                                        if base_b else None)
    if df_c is not None:
        pos = df_c[df_c["has_vuln"] == True]
        d["vuln_keyword_hits"] = f"{int(pos['mot_cle_trouve'].fillna(False).sum())}/{len(pos)}"
        d["rows_to_grade_C"] = int((~df_c["same_as_ref"]).sum())
    return d

for variant in VARIANTS:
    tag = f"{model_slug}__{MODEL_KIND}__{variant}"
    if not OVERWRITE and os.path.exists(f"{OUT}/summary_{tag}.json"):
        print(f"== {variant} already done, skipping ==")
        continue
    print(f"\n========== {MODEL_KIND} / {variant} ==========")
    model = load_model(variant)
    weights_gb = model.get_memory_footprint() / 1e9
    print(f"Weights in memory: {weights_gb:.2f} GB")

    t = generate(model, "Write a Python function add(a, b) that returns a + b.")     # warm-up + sanity check
    print(t["response"][:200])
    assert "def" in t["response"], f"Incoherent answer with variant {variant}"

    df_a = run_part_a(model, variant) if "A" in PARTS else None
    he = run_part_b(model, variant) if "B" in PARTS else None
    df_c = run_part_c(model, variant) if "C" in PARTS else None
    if df_a is not None: df_a.to_csv(f"{OUT}/benchmark_{tag}.csv", index=False)
    if he is not None:
        with open(f"{OUT}/humaneval_{tag}.json", "w", encoding="utf-8") as f:
            json.dump(he, f, ensure_ascii=False, indent=2)
    if df_c is not None: df_c.to_csv(f"{OUT}/vulnerabilites_{tag}.csv", index=False)
    with open(f"{OUT}/summary_{tag}.json", "w") as f:
        json.dump(summarize(variant, df_a, he, df_c, weights_gb), f, indent=2)

    del model
    gc.collect(); torch.cuda.empty_cache()
print("\nDone.")


========== ft / fp16 ==========


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Weights in memory: 3.09 GB
```python
def add(a, b):
    return a + b
```

The `add` function takes two parameters, `a` and `b`, and returns their sum. This is a simple addition operation in Python.
fp16 P1 pass | identique bf16: False
5
fp16 P2 pass | identique bf16: False
monde le Bonjour
fp16 P3 pass | identique bf16: False
3
fp16 P4 pass | identique bf16: False
fp16 P5 fail | identique bf16: True
fp16 P6 pass | identique bf16: True
fp16 P7 None | identique bf16: False
fp16 P8 None | identique bf16: True
fp16 P9 None | identique bf16: False
fp16 P10 None | identique bf16: False
fp16 P11 None | identique bf16: False
fp16 P12 None | identique bf16: False
fp16 P13 None | identique bf16: False
fp16 P14 None | identique bf16: False
fp16 P15 None | identique bf16: False
fp16 HumanEval/0 pass
fp16 HumanEval/1 pass
fp16 HumanEval/2 pass
fp16 HumanEval/3 pass
fp16 HumanEval/4 pass
fp16 HumanEval/5 pass
fp16 HumanEval/6 fail
fp16 HumanEval/7 pass
fp16 HumanEval/8 pass
fp16 HumanEval/9 pass
fp1

/tmp/ipykernel_50/1692323391.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d["vuln_keyword_hits"] = f"{int(pos['mot_cle_trouve'].fillna(False).sum())}/{len(pos)}"



========== ft / int8 ==========


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Weights in memory: 1.12 GB
```python
def add(a, b):
    return a + b
```

The `add` function takes two parameters, `a` and `b`, and returns their sum using the addition operator `+`. This is a simple example of a function in Py
nf4 P1 pass | identique bf16: False
5
nf4 P2 pass | identique bf16: False
world! Hello
nf4 P3 pass | identique bf16: False
3
nf4 P4 pass | identique bf16: False
nf4 P5 pass | identique bf16: False
nf4 P6 pass | identique bf16: False
nf4 P7 None | identique bf16: False
nf4 P8 None | identique bf16: False
nf4 P9 None | identique bf16: False
nf4 P10 None | identique bf16: False
nf4 P11 None | identique bf16: False
nf4 P12 None | identique bf16: False
nf4 P13 None | identique bf16: False
nf4 P14 None | identique bf16: False
nf4 P15 None | identique bf16: False
nf4 HumanEval/0 pass
nf4 HumanEval/1 pass
nf4 HumanEval/2 pass
nf4 HumanEval/3 pass
nf4 HumanEval/4 pass
nf4 HumanEval/5 pass
nf4 HumanEval/6 pass
nf4 HumanEval/7 pass
nf4 HumanEval/8 pass
nf4 HumanEval/9 pass

/tmp/ipykernel_50/1692323391.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d["vuln_keyword_hits"] = f"{int(pos['mot_cle_trouve'].fillna(False).sum())}/{len(pos)}"



Done.


In [8]:
rows = []
for variant in VARIANTS:
    path = f"{OUT}/summary_{model_slug}__{MODEL_KIND}__{variant}.json"
    if os.path.exists(path):
        rows.append(json.load(open(path)))
table = pd.DataFrame(rows).set_index("variant")
if "fp16" in table.index:
    table["speed_vs_fp16"] = (table["tokens_per_sec"] / table.loc["fp16", "tokens_per_sec"]).round(2)
    table["peak_mem_vs_fp16"] = (table["peak_gpu_gb"] / table.loc["fp16", "peak_gpu_gb"]).round(2)
table.to_csv(f"{OUT}/comparatif_quantification_{MODEL_KIND}_{model_slug}.csv")
table.T

variant,fp16,int8,nf4
model_kind,ft,ft,ft
weights_gb,3.09,1.78,1.12
peak_gpu_gb,3.12,1.82,1.2
ttft_s,0.047,0.193,0.068
latency_s,6.02,25.2,11.63
tokens_per_sec,26.17,6.61,19.0
output_tokens,157.9,166.5,221.1
auto_pass_P1_P6,5/6,5/6,6/6
same_text_as_ref_A_%,20.0,6.7,0.0
rows_to_grade_A,12,14,15


In [9]:
from importlib.metadata import version
meta = {"model_kind": MODEL_KIND, "parts": PARTS, "variants": VARIANTS, "model": MODEL_NAME, "gpu": GPU_NAME,
        "adapter": ADAPTER_META, "generation_params": GEN_PARAMS,
        "merge": "adapter merged into fp16 weights, then quantized at load time" if MODEL_KIND == "ft" else None,
        "nf4_config": {"quant_type": "nf4", "double_quant": True, "compute_dtype": "float16"},
        "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
        "versions": {k: version(k) for k in ["transformers", "accelerate", "bitsandbytes", "peft", "datasets", "pandas"]},
        "note": "torchao uninstalled (preinstalled 0.10.0 incompatible with peft 0.21.2)"}
json.dump(meta, open(f"{OUT}/run_meta_quant_{MODEL_KIND}.json", "w"), indent=2)
out_zip = shutil.make_archive(f"/kaggle/working/quant_{MODEL_KIND}_results", "zip", OUT)
print(json.dumps(meta, indent=2)); print("\nTo download:", out_zip)

{
  "model_kind": "ft",
  "parts": [
    "A",
    "B",
    "C"
  ],
  "variants": [
    "fp16",
    "int8",
    "nf4"
  ],
  "model": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "gpu": "Tesla T4",
  "adapter": {
    "r": 16,
    "lora_alpha": 16,
    "sha256": "02ec15117c3bb9e0e84b5a0a7eec98f0bf1d14f438d6fa592d56a3cbe90ab9f3"
  },
  "generation_params": {
    "max_new_tokens": 512,
    "do_sample": false
  },
  "merge": "adapter merged into fp16 weights, then quantized at load time",
  "nf4_config": {
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "versions": {
    "transformers": "5.20.2",
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.2",
    "peft": "0.21.2",
    "datasets": "4.8.5",
    "pandas": "2.3.3"
  },
  "note": "torchao uninstalled (preinstalled 0.10.0 incompatible with peft 0.21.2)"
}

To download: /kaggle/working/quant_ft_results.zip
